### Tools
- **Model**   : `mistralai/Mistral-7B-Instruct-v0.3` (free via Hugging Face)
- **Library** : `openai` SDK pointed at Hugging Face router (official 2025 method)
- **API URL** : `https://router.huggingface.co/v1` (current correct endpoint)
- **Safety**  : Emergency keyword filter + system-prompt guardrails

In [1]:
!pip install openai --quiet
print("Done.")

Done.


In [2]:
from openai import OpenAI

print("Library imported successfully.")

Library imported successfully.


In [3]:
# Hugging Face Token
# Get your free token from: https://huggingface.co/settings/tokens
HF_TOKEN = "your-huggingface-token-here" 

client = OpenAI(
    base_url="https://router.huggingface.co/v1",
    api_key=HF_TOKEN
)

# Model to use — free Mistral 7B Instruct
MODEL = "meta-llama/Llama-3.1-8B-Instruct:cerebras"

print("Client initialized successfully.")
print(f"Model   : {MODEL}")
print(f"API URL : https://router.huggingface.co/v1")

Client initialized successfully.
Model   : meta-llama/Llama-3.1-8B-Instruct:cerebras
API URL : https://router.huggingface.co/v1


## Prompt Engineering

In [4]:
SYSTEM_PROMPT = """You are a friendly and knowledgeable general health assistant.
Act like a helpful medical assistant who explains things in simple, clear language.

Your responsibilities:
- Answer general health questions clearly and compassionately.
- Explain medical terms in plain language anyone can understand.
- Provide evidence-based general health information.
- Always recommend consulting a real doctor for personal medical decisions.

STRICT SAFETY RULES (never break these):
1. Never diagnose any medical condition for the user.
2. Never recommend specific dosages for the user personally.
3. If the user describes an emergency (chest pain, difficulty breathing,
   severe bleeding), tell them to call emergency services immediately.
4. End every response about symptoms or medications with a reminder to see a doctor.
5. Never act as a replacement for professional medical advice.
6. Keep answers concise: 3 to 5 sentences for simple questions.
7. Always use an empathetic, warm, and supportive tone."""

print("System prompt defined.")
print(f"Length: {len(SYSTEM_PROMPT)} characters")

System prompt defined.
Length: 996 characters


## Safety Filter (Emergency Keyword Detection)

In [5]:
EMERGENCY_KEYWORDS = [
    "chest pain",
    "can't breathe",
    "cannot breathe",
    "difficulty breathing",
    "not breathing",
    "stroke",
    "unconscious",
    "overdose",
    "severe bleeding",
    "heart attack",
    "poisoning"
]

EMERGENCY_RESPONSE = (
    "MEDICAL EMERGENCY DETECTED.\n"
    "Please call emergency services IMMEDIATELY:\n"
    "  Pakistan  : 1122 (Rescue) or 115 (Ambulance)\n"
    "  UK        : 999\n"
    "  USA/Canada: 911\n"
    "Do not wait. Go to the nearest emergency room right away."
)


def is_emergency(text):
    """Return True if text contains any emergency keyword."""
    return any(kw in text.lower() for kw in EMERGENCY_KEYWORDS)


# Verify the filter works
tests = [
    ("I have chest pain",             True),
    ("What causes a headache?",       False),
    ("I cannot breathe properly",     True),
    ("Is paracetamol safe for kids?", False),
]
print("Safety filter verification:")
for text, expected in tests:
    result = is_emergency(text)
    tag = "PASS" if result == expected else "FAIL"
    print(f"  [{tag}]  '{text}'  ->  Emergency={result}")

Safety filter verification:
  [PASS]  'I have chest pain'  ->  Emergency=True
  [PASS]  'What causes a headache?'  ->  Emergency=False
  [PASS]  'I cannot breathe properly'  ->  Emergency=True
  [PASS]  'Is paracetamol safe for kids?'  ->  Emergency=False


## Core Chat Function

In [6]:
def chat(query, history):
    """
    Send a health query to Mistral-7B via Hugging Face and return the response.

    Parameters
    ----------
    query   : str   -- user's health question
    history : list  -- list of {role, content} message dicts

    Returns
    -------
    reply   : str   -- assistant response text
    history : list  -- updated conversation history
    """
    # Layer 1: Emergency keyword check — no API call needed
    if is_emergency(query):
        return EMERGENCY_RESPONSE, history

    # Add user message to history
    history.append({"role": "user", "content": query})

    # Build messages: system prompt + full conversation history
    messages = [{"role": "system", "content": SYSTEM_PROMPT}] + history

    # Call Hugging Face via OpenAI-compatible router
    response = client.chat.completions.create(
        model=MODEL,
        messages=messages,
        max_tokens=500,
        temperature=0.7
    )

    # Extract the reply text
    reply = response.choices[0].message.content.strip()

    # Save assistant reply into history for next turn
    history.append({"role": "assistant", "content": reply})

    return reply, history


print("chat() function ready.")

chat() function ready.


## Connection Test

In [7]:
print("Testing connection to Hugging Face...")
print()

try:
    test_reply, _ = chat("Say hello in one sentence.", [])
    print("RESPONSE :", test_reply)
    print()
    print("Connection successful! Proceed to Step 8.")
except Exception as e:
    print(f"ERROR: {e}")
    print()
    print("Most likely cause: invalid HF_TOKEN in Step 3.")
    print("Fix: replace HF_TOKEN with your real token from https://huggingface.co/settings/tokens")

Testing connection to Hugging Face...

RESPONSE : Hello, I'm here to help answer any general health questions you may have, and I'm looking forward to supporting you on your health journey.

Connection successful! Proceed to Step 8.


## Test with Queries from Task Specification

In [8]:
example_queries = [
    "What causes a sore throat?",           
    "Is paracetamol safe for children?",    
    "How can I manage stress better?",
    "What are the symptoms of dehydration?"
]

history = []   # Fresh conversation

for query in example_queries:
    print("=" * 65)
    print(f"USER      : {query}")
    print("-" * 65)
    reply, history = chat(query, history)
    print(f"ASSISTANT : {reply}")
    print()

USER      : What causes a sore throat?
-----------------------------------------------------------------
ASSISTANT : A sore throat can be caused by a viral infection, such as a cold or flu, or by an allergic reaction to something like dust, pollen, or certain foods. It can also be caused by overusing your voice, dry air, or bacterial infections like strep throat. Sometimes, a sore throat can even be a sign of a more serious underlying condition, like a sinus infection or tonsillitis. If you're experiencing a sore throat, it's always best to see a doctor to determine the cause and get proper treatment.

USER      : Is paracetamol safe for children?
-----------------------------------------------------------------
ASSISTANT : Paracetamol can be safe for children when used correctly, but it's essential to follow the dosage instructions carefully. Always check with your pediatrician or the child's doctor before giving paracetamol to a child, and never exceed the recommended dose. You shoul

## Test Emergency Safety Filter

In [9]:
# This does NOT call the API — handled by keyword filter locally
emergency_q = "I am having severe chest pain and difficulty breathing"

print("=" * 65)
print(f"USER      : {emergency_q}")
print("-" * 65)
reply, _ = chat(emergency_q, [])
print(f"ASSISTANT :\n{reply}")

USER      : I am having severe chest pain and difficulty breathing
-----------------------------------------------------------------
ASSISTANT :
MEDICAL EMERGENCY DETECTED.
Please call emergency services IMMEDIATELY:
  Pakistan  : 1122 (Rescue) or 115 (Ambulance)
  UK        : 999
  USA/Canada: 911
Do not wait. Go to the nearest emergency room right away.


## Interactive Live Chatbot

Type your questions and press Enter. Type quit to stop.

In [10]:
def run_chatbot():
    print("=" * 65)
    print("        GENERAL HEALTH QUERY CHATBOT")
    print("        Powered by Mistral-7B via Hugging Face (Free)")
    print("=" * 65)
    print("Disclaimer: General health information only.")
    print("NOT a substitute for professional medical advice.")
    print("Type 'quit' or 'exit' to end the session.")
    print("=" * 65)

    conv_history = []

    while True:
        try:
            user_input = input("\nYou: ").strip()
        except (EOFError, KeyboardInterrupt):
            print("\nSession ended.")
            break

        if not user_input:
            print("Please type a question.")
            continue

        if user_input.lower() in ["quit", "exit", "bye"]:
            print("\nAssistant: Stay healthy! See a doctor for any serious concerns. Goodbye!")
            break

        try:
            reply, conv_history = chat(user_input, conv_history)
            print(f"\nAssistant: {reply}")
        except Exception as err:
            print(f"\nError: {err}")
            print("Check your HF_TOKEN and internet connection, then try again.")


run_chatbot()

        GENERAL HEALTH QUERY CHATBOT
        Powered by Mistral-7B via Hugging Face (Free)
Disclaimer: General health information only.
NOT a substitute for professional medical advice.
Type 'quit' or 'exit' to end the session.



You:  How can i manage stress better?



Assistant: Managing stress is a great goal to work towards. There are many ways to reduce stress, but here are a few simple tips that can make a big difference:

1. **Take deep breaths**: When we're stressed, our breathing gets shallow. Consciously taking a few deep breaths can calm our body and mind.
2. **Exercise regularly**: Physical activity releases endorphins, which are natural mood-boosters. Even a short walk each day can help!
3. **Practice mindfulness**: Focus on the present moment, without judgment. You can try meditation, yoga, or simply paying attention to your senses (sights, sounds, smells, etc.).
4. **Get enough sleep**: Aim for 7-8 hours of sleep each night to help your body and mind recharge.
5. **Set boundaries**: Learn to say "no" to things that drain your energy, and prioritize activities that bring you joy.

Remember, everyone is unique, so experiment with different techniques to find what works best for you. And most importantly, **see a doctor** if you're strugg


You:  quit



Assistant: Stay healthy! See a doctor for any serious concerns. Goodbye!


## Results & Key Findings

### System Architecture
```
User Input
    |
    v
[Safety Filter] -- emergency keyword? --> YES --> Local response, no API call
    |
    NO
    |
    v
[OpenAI-compatible client]
    base_url = https://router.huggingface.co/v1
    model    = mistralai/Mistral-7B-Instruct-v0.3
    messages = [system prompt] + conversation history
    |
    v
Response --> Shown to user --> Saved to history for next turn
```

### Prompt Engineering Techniques Used

| Technique | Applied As |
|---|---|
| Persona assignment | "Act like a helpful medical assistant" |
| Numbered safety rules | 7 hard constraints the model cannot break |
| Tone control | "empathetic, warm, and supportive" |
| Length control | "3 to 5 sentences for simple questions" |
| Doctor reminder | Always recommends consulting a real doctor |

### Key Findings
- Prompt engineering shapes safe, professional behavior without any fine-tuning
- Two-layer safety (keyword filter + LLM rules) is more robust than either alone
- Hugging Face's OpenAI-compatible router is the correct 2025 method for LLM inference
- Conversation history enables context-aware multi-turn follow-up questions
- Mistral-7B is completely free and highly capable for general health Q&A